# Risk & factor analytics with `Performance`

**Purpose:** Use one `Performance` object to compute return, risk, tail, benchmark-relative, rolling-regression, and factor-model analytics.

**Prerequisites:** `03_analytics/performance_analytics.ipynb`.

**What you'll learn:**

- Build a small fund/benchmark return panel from the shared synthetic fixture.
- Inspect benchmark regression and risk diagnostics.
- Read per-period returns back out of `Performance` with `returns()` / `returns_for_ticker()`.
- Connect return-based factor analytics to later portfolio factor-sensitivity notebooks.

Every analytic in this notebook flows through a single `Performance` instance: construct it once from prices or returns, then call methods for the views you need.


## Build a small panel

We simulate two correlated daily return series — a fund and a benchmark — over one trading year, then construct a single `Performance` for the panel.

The synthetic walk comes from `_shared.random_walk_panel`, the example-suite fixture generator: it owns the business-day calendar and the seeded RNG so every notebook draws its toy data the same way. The fund is deliberately built as a **single-index** series (`fund = β·bench + residual`) so the regressions further down have something to recover.

Note the constructor: `Performance.from_returns_arrays` takes per-period returns directly. There is no reason to compound returns into a price path in Python only to have Rust difference them again.


In [1]:
import sys
sys.path.insert(0, "..")

from _shared import random_walk_panel

from finstack_quant.analytics import Performance

# `_shared.random_walk_panel` supplies the business-day axis (pd.bdate_range) and a
# single deterministic seeded stream. Two independent driver walks: the benchmark
# itself, and an idiosyncratic series that becomes the fund's residual.
BETA = 0.88
panel = random_walk_panel(
    253,  # 253 prices -> 252 return observations, i.e. one trading year
    2,
    seed=42,
    drift=[0.0004, 0.00025],  # benchmark drift, fund alpha
    vol=[0.01, 0.0035],  # benchmark vol, residual vol
    names=["BENCH", "IDIO"],
)
bench_returns, idio_returns = panel.returns

# Single-index fixture: fund return = BETA * benchmark + idiosyncratic residual.
fund_returns = [BETA * b + e for b, e in zip(bench_returns, idio_returns, strict=True)]

# Hand the raw returns to Rust rather than compounding prices in Python; the
# return dates are panel.dates[1:] because each return spans a pair of prices.
perf = Performance.from_returns_arrays(
    panel.dates[1:],
    [fund_returns, bench_returns],
    ["FUND", "BENCH"],
    benchmark_ticker="BENCH",
)
print("Tickers:", perf.ticker_names)
print("Bench idx:", perf.benchmark_idx)
print("Active dates:", len(perf.dates()))
print(f"Designed beta: {BETA:.2f} (the regressions below should recover roughly this)")


Tickers: ['FUND', 'BENCH']
Bench idx: 1
Active dates: 252
Designed beta: 0.88 (the regressions below should recover roughly this)


## Single-factor benchmark regression

`Performance.beta()` returns OLS beta with standard error and a 95% CI per ticker; `greeks()` returns annualized alpha, beta, and R². Both use the panel's designated benchmark.


In [2]:
betas = perf.beta()
gks = perf.greeks()

fund_beta = betas[0]
fund_gr = gks[0]
print(f"Beta: {fund_beta.beta:.4f} (SE {fund_beta.std_err:.4f})")
print(f"95% CI: [{fund_beta.ci_lower:.4f}, {fund_beta.ci_upper:.4f}]")
print(f"Greeks alpha (ann.): {fund_gr.alpha:.6f}")
print(f"Greeks beta: {fund_gr.beta:.4f}; R²: {fund_gr.r_squared:.4f}")

Beta: 0.8730 (SE 0.0230)
95% CI: [0.8279, 0.9181]
Greeks alpha (ann.): 0.045708
Greeks beta: 0.8730; R²: 0.8519


## Rolling greeks

`Performance.rolling_greeks(ticker_idx, window)` fits the single-index model on a moving window and returns time series of alpha and beta aligned to right-labelled window-end dates.


In [3]:
rg = perf.rolling_greeks(0, window=63)
print(f"Rolling windows: {len(rg.alphas)}")
print(f"Last alpha: {rg.alphas[-1]:.6f}; last beta: {rg.betas[-1]:.4f}")

Rolling windows: 190
Last alpha: -0.109975; last beta: 0.8733


## VaR, ES, and tail diagnostics

`Performance` exposes historical, parametric, and Cornish–Fisher VaR plus expected shortfall, as well as higher moments (`skewness`, `kurtosis`) and the asymmetric `tail_ratio`.


In [4]:
conf = 0.95
print(f"Historical VaR ({conf:.0%}): {perf.value_at_risk(conf)[0]:.6f}")
print(f"Parametric VaR ({conf:.0%}): {perf.parametric_var(conf)[0]:.6f}")
print(f"Cornish–Fisher VaR ({conf:.0%}): {perf.cornish_fisher_var(conf)[0]:.6f}")
print(f"Expected shortfall ({conf:.0%}): {perf.expected_shortfall(conf)[0]:.6f}")
print(f"Skewness: {perf.skewness()[0]:.4f}")
print(f"Kurtosis: {perf.kurtosis()[0]:.4f}")
print(f"Tail ratio: {perf.tail_ratio()[0]:.4f}")

Historical VaR (95%): -0.013842
Parametric VaR (95%): -0.013762
Cornish–Fisher VaR (95%): -0.014054
Expected shortfall (95%): -0.018187
Skewness: -0.1342
Kurtosis: 0.3035
Tail ratio: 1.2378


## Benchmark-relative metrics

Up/down capture, capture ratio, tracking error, information ratio, R², and batting average — all per-ticker against the panel benchmark.


In [5]:
print(f"Up capture: {perf.up_capture()[0]:.4f}")
print(f"Down capture: {perf.down_capture()[0]:.4f}")
print(f"Capture ratio: {perf.capture_ratio()[0]:.4f}")
print(f"Tracking error: {perf.tracking_error()[0]:.6f}")
print(f"Information ratio: {perf.information_ratio()[0]:.4f}")
print(f"R-squared: {perf.r_squared()[0]:.4f}")
print(f"Batting average: {perf.batting_average()[0]:.4f}")

Up capture: 0.7414
Down capture: 0.9264
Capture ratio: 0.8003
Tracking error: 0.059722
Information ratio: 0.0047
R-squared: 0.8519
Batting average: 0.4762


## Drawdown episodes

`drawdown_details(ticker_idx, n=...)` returns the deepest drawdown episodes for a single ticker with start, valley, and recovery dates.


In [6]:
print(f"Max drawdown (fund): {perf.max_drawdown()[0]:.2%}")
episodes = perf.drawdown_details(0, n=3)
for i, ep in enumerate(episodes, start=1):
    end = ep.end
    end_s = end.isoformat() if end else "(ongoing)"
    print(
        f"  #{i} Start: {ep.start.isoformat()}, Valley: {ep.valley.isoformat()}, "
        f"End: {end_s}, Depth: {ep.max_drawdown:.2%}"
    )

Max drawdown (fund): -6.33%
  #1 Start: 2024-01-18, Valley: 2024-02-19, End: 2024-03-25, Depth: -6.33%
  #2 Start: 2024-11-20, Valley: 2024-12-05, End: (ongoing), Depth: -5.68%
  #3 Start: 2024-04-11, Valley: 2024-04-29, End: 2024-05-08, Depth: -4.24%


## Multi-factor model

`Performance.multi_factor_greeks(ticker_idx, factor_returns)` fits a multivariate OLS against several aligned factor return series and returns alpha, betas, R², adjusted R², and residual volatility.

The market factor here is the panel's own benchmark, so take its returns from `Performance.returns_for_ticker(idx)` — the canonical per-period return accessor (`returns()` gives the whole panel). Do **not** rebuild them by un-compounding `cumulative_returns()`: that reintroduces return-linking arithmetic in Python, and any disagreement with the Rust convention shows up silently as a wrong beta.


In [7]:
import random

active_dates = perf.dates()
n_active = len(active_dates)
random.seed(7)
smb_returns = [random.gauss(0.0, 0.005) for _ in range(n_active)]
hml_returns = [random.gauss(0.0, 0.004) for _ in range(n_active)]

# The market factor is simply the benchmark's own per-period returns. Take them
# from the canonical accessor -- never re-derive them by un-compounding
# `cumulative_returns()`, and no longer via the `excess_returns([0.0] * n)` trick.
market_returns = perf.returns_for_ticker(perf.benchmark_idx)

mf = perf.multi_factor_greeks(0, [market_returns, smb_returns, hml_returns])
print(f"Alpha (ann.): {mf.alpha:.6f}")
print(f"Betas [mkt, size, value]: {[round(b, 4) for b in mf.betas]}")
print(f"R²: {mf.r_squared:.4f}; Adj R²: {mf.adjusted_r_squared:.4f}")
print(f"Residual vol: {mf.residual_vol:.6f}")

# `returns()` is the panel-shaped sibling: one series per ticker, same convention.
panel_returns = perf.returns()
print(f"\nreturns() panel: {len(panel_returns)} series x {len(panel_returns[perf.benchmark_idx])} periods")

Alpha (ann.): 0.049927
Betas [mkt, size, value]: [np.float64(0.8724), np.float64(-0.0631), np.float64(0.0253)]
R²: 0.8532; Adj R²: 0.8515
Residual vol: 0.056472

returns() panel: 2 series x 252 periods


## Periodic returns

`lookback_returns` reports period-to-date returns (MTD / QTD / YTD, and FYTD when a fiscal year start month is provided). `rolling_returns(ticker_idx, window)` compounds returns over a rolling window for trailing 1Y / 3Y views.


In [8]:
ref = perf.dates()[-1]
lb = perf.lookback_returns(ref, fiscal_year_start_month=4)
print(f"As of {ref.isoformat()}:")
print(f"  MTD: {lb.mtd[0]:.4f}")
print(f"  QTD: {lb.qtd[0]:.4f}")
print(f"  YTD: {lb.ytd[0]:.4f}")
if lb.fytd is not None:
    print(f"  FYTD (Apr): {lb.fytd[0]:.4f}")

roll_1y = perf.rolling_returns(0, window=252)
print(f"Trailing 1Y rolling returns: {len(roll_1y.values)} windows")
if len(roll_1y.values) > 0:
    print(f"  Last window end: {roll_1y.dates[-1].isoformat()}")
    print(f"  Last 1Y compounded return: {roll_1y.values[-1]:.4f}")

As of 2024-12-18:
  MTD: -0.0034
  QTD: 0.0506
  YTD: 0.4149
  FYTD (Apr): 0.3619
Trailing 1Y rolling returns: 1 windows
  Last window end: 2024-12-18
  Last 1Y compounded return: 0.4149


## Takeaways

- `Performance` is the single binding entry point: construct from prices (or returns) and reach every analytic as a method.
- Benchmark regressions (`beta`, `greeks`, `rolling_greeks`) and multi-factor models (`multi_factor_greeks`) cover the alpha / beta / R² questions.
- Need the raw per-period returns back? Use `returns()` (panel) or `returns_for_ticker(idx)` (single series). Un-compounding `cumulative_returns()` in Python is return-linking logic that belongs in Rust.
- Tail risk (`value_at_risk`, `expected_shortfall`, parametric / Cornish–Fisher variants, higher moments, `tail_ratio`) and benchmark-relative metrics live on the same instance.
- Periodic and rolling views (`lookback_returns`, `rolling_returns`, `rolling_sharpe`, `rolling_volatility`) answer MTD / QTD / YTD / FYTD and trailing-window questions directly.
